In [1]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [3]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),Available,Battery Capacity,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_EG,Log_Available,Log_AC Recharging Speed (km/h)
0,Austria,2020,6.41,3.803179e+05,1.381911,-1.600,26797.00000,39.325301,403.164557,97.000000,57.761446,286.200000,38311.380000,2.002830,10.196083,12.848765,0.867903,NaN,4.584967,3.696979
1,Austria,2021,13.92,4.062315e+05,2.766667,2.000,39904.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,2.702703,10.594257,12.914681,1.326190,1.098612,5.129899,3.735390
2,Austria,2022,15.70,4.493822e+05,8.546870,2.700,61393.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,2.815409,11.025067,13.015631,2.256213,1.308333,5.537334,3.918300
3,Austria,2023,19.91,4.778373e+05,7.814134,0.800,76773.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.040228,11.248621,13.077028,2.176357,0.587787,5.918894,4.267004
4,Austria,2024,17.58,4.940876e+05,2.937916,0.100,102562.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,2.922086,11.538233,13.110470,1.370652,0.095310,6.314133,4.534494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,Sweden,2021,19.09,5.417760e+06,2.163197,1.300,70537.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,3.000222,11.163907,15.505193,1.151583,0.832909,5.129899,3.735390
158,Sweden,2022,32.84,5.816415e+06,8.369291,3.500,83510.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,3.521644,11.332734,15.576195,2.237437,1.504077,5.537334,3.918300
159,Sweden,2023,38.42,6.143187e+06,8.548625,1.200,126758.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.674273,11.750043,15.630854,2.256397,0.788457,5.918894,4.267004
160,Sweden,2024,34.99,6.379843e+06,2.835817,-0.300,189977.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,3.583241,12.154664,15.668654,1.344382,-0.356675,6.314133,4.534494


In [4]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [5]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Data splitting into train data and test data
pre_covid_df = dataset[dataset['Year'].between(2020, 2024)].copy()
post_covid_df = dataset[dataset['Year'] >= 2025].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [7]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)         0
Log_Recharging Points              1
Real Range                         0
Purchase price (EUR)               0
Log_Available                      0
Log_AC Recharging Speed (km/h)     0
Battery Capacity                   0
Log_GDP                            0
Log_CPI                            1
Log_EG                            22
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [8]:
# Checking which rows are empty
print("Missing Log_EG in Train data (2020-2024)")
print(pre_covid_df[pre_covid_df['Log_EG'].isna()].reset_index()[['Country', 'Year', 'Log_EG']])

Missing Log_EG in Train data (2020-2024)
           Country  Year  Log_EG
0          Austria  2020     NaN
1         Bulgaria  2020     NaN
2          Croatia  2020     NaN
3   Czech Republic  2020     NaN
4          Denmark  2020     NaN
5          Estonia  2020     NaN
6          Finland  2020     NaN
7          Finland  2024     NaN
8           Greece  2020     NaN
9          Hungary  2020     NaN
10         Hungary  2024     NaN
11         Ireland  2020     NaN
12           Italy  2020     NaN
13          Latvia  2021     NaN
14          Latvia  2024     NaN
15       Lithuania  2020     NaN
16        Portugal  2020     NaN
17         Romania  2020     NaN
18         Romania  2023     NaN
19        Slovakia  2020     NaN
20           Spain  2020     NaN
21          Sweden  2020     NaN


In [9]:
features_to_impute = ['Log_Recharging Points', 'Log_EG', 'Log_CPI']

# Compute per-country median from train
train_country_medians = {}
train_global_medians = {}

for col in features_to_impute:
    train_country_medians[col] = pre_covid_df.groupby('Country')[col].median()
    train_global_medians[col] = pre_covid_df[col].median()

def robust_impute(df, country_medians, global_medians):
    df_copy = df.copy()
    for col in features_to_impute:

        # Impute per-country median
        df_copy[col] = df_copy[col].fillna(df_copy['Country'].map(country_medians[col]))

        # Fallback to global median if still NaN
        df_copy[col] = df_copy[col].fillna(global_medians[col])
    return df_copy

# Apply to train and test
pre_covid_fixed = robust_impute(pre_covid_df, train_country_medians, train_global_medians)
post_covid_fixed = robust_impute(post_covid_df, train_country_medians, train_global_medians)

In [10]:
# Verfication
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [11]:
X_train_scaled = scaler.transform(pre_covid_fixed[features])
X_test_scaled = scaler.transform(post_covid_fixed[features])

# Convert back to DataFrame to keep column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features)

In [12]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


In [13]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (135, 3)
Test Shape (PCs): (27, 3)


In [14]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    "Austria", "Belgium", "Czech Republic", "Denmark", "France",
    "Germany", "Hungary", "Italy", "Netherlands", "Poland",
    "Romania", "Spain", "Sweden"
]

followers_list = [
    "Bulgaria", "Croatia", "Cyprus", "Estonia", "Finland",
    "Greece", "Ireland", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Portugal", "Slovakia", "Slovenia"
]

In [15]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Neural Network Models

**1. LSTM Model**

In [16]:
def run_lstm_model(X_train, y_train, X_test, y_test, cluster_name):
    # Scaling
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    y_train_arr = y_train.values.reshape(-1, 1)

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train_arr)

    # Reshape to 3D
    X_train_3D = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3D = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    # Build Architecture - Trial 3 adds more capacity
    model = Sequential([
        LSTM(128, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=True),
        Dropout(0.1),
        LSTM(64, activation='tanh', return_sequences=False),
        Dropout(0.1),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Callbacks
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True
    )

    # Fit the model
    model.fit(
        X_train_3D, y_train_scaled,
        epochs=500,
        batch_size=4,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    # Predict and Inverse Scale
    train_pred_scaled = model.predict(X_train_3D)
    test_pred_scaled = model.predict(X_test_3D)

    train_pred_log = scaler_y.inverse_transform(train_pred_scaled).flatten()
    test_pred_log = scaler_y.inverse_transform(test_pred_scaled).flatten()

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred_log)
    test_pred_real = np.exp(test_pred_log)

    # Compute Metrics
    r2_real = r2_score(y_test_real, test_pred_real)
    mae_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - LSTM Trial 3")
    print(f"Test R² (Real): {r2_real:.4f}, MAE (Real): {mae_real:.2f}%\n")

    return {
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [17]:
# Run for both clusters
results_lstm_leaders = run_lstm_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_lstm_followers = run_lstm_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Leaders - LSTM Trial 3
Test R² (Real): 0.0332, MAE (Real): 12.42%



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Followers - LSTM Trial 3
Test R² (Real): -0.6067, MAE (Real): 9.42%



In [18]:
# Combine predictions and actual values
combined_preds = np.concatenate([
    results_lstm_leaders['pred_test_real'],
    results_lstm_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_lstm_leaders['y_test_real'],
    results_lstm_followers['y_test_real']
])

# Combine metadata in the same order (ensuring X_test slice matches post_covid_fixed order)
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_lstm_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error calculation
all_lstm_results['Error_Abs'] = (all_lstm_results['Actual_BEV_%'] - all_lstm_results['Predicted_BEV_%']).abs()

# Print results by year and cluster
for year in sorted(all_lstm_results['Year'].unique()):
    print(f"\n---> LSTM: {int(year)} PREDICTION RESULTS")

    year_data = all_lstm_results[all_lstm_results['Year'] == year]

    # Leaders Display
    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Display
    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))


---> LSTM: 2025 PREDICTION RESULTS

**** LEADERS CLUSTER (2025) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         69.53        18.739092  50.790908
   Netherlands         37.68        18.408869  19.271131
        Sweden         37.58        21.164646  16.415354
       Belgium         35.40        18.980341  16.419659
       Austria         22.34        17.860651   4.479349
        France         21.01        20.019779   0.990221
       Germany         20.08        21.033907   0.953907
         Spain         10.19        16.324486   6.134486
       Hungary          9.36        19.060165   9.700165
        Poland          7.73        13.585986   5.855986
         Italy          7.18        17.690062  10.510062
       Romania          6.59        14.975598   8.385598
Czech Republic          6.55        18.067577  11.517577

**** FOLLOWERS CLUSTER (2025) ****
   Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Finland         37.81         7.479280  

In [19]:
# Save the results in an excel file

# Filter the results as per the clusters
all_lstm_results['Cluster'] = all_lstm_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial3_LSTM_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial3_LSTM_Followers_Results.xlsx", index=False)